# Chest X-Ray Dataset — Exploratory Data Analysis

**Dataset:** Kaggle Chest X-Ray Images (Pneumonia)
**Classes:** NORMAL, PNEUMONIA
**Purpose:** Complete EDA study — no model training

---

## 1. Setup & Imports

In [1]:
import os, sys, hashlib, csv, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter, ImageStat

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Paths — resolve from notebook location
NOTEBOOK_DIR = Path(".").resolve()
# Support running from hms-ai root or from ml/eda/
if (NOTEBOOK_DIR / "app" / "data" / "chest_xray").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent.parent / "app" / "data" / "chest_xray").exists():
    PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
else:
    raise FileNotFoundError("Cannot find app/data/chest_xray from current directory")

DATA_DIR = PROJECT_ROOT / "app" / "data" / "chest_xray"
OUTPUT_DIR = PROJECT_ROOT / "ml" / "eda" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = {"train": DATA_DIR / "train", "val": DATA_DIR / "val", "test": DATA_DIR / "test"}
CLASSES = ["NORMAL", "PNEUMONIA"]
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

print(f"Dataset:  {DATA_DIR}")
print(f"Outputs:  {OUTPUT_DIR}")
print(f"Python:   {sys.version}")


Dataset:  C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-project\hms-ai\app\data\chest_xray
Outputs:  C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-project\hms-ai\ml\eda\outputs
Python:   3.11.4 (tags/v3.11.4:d2340ef, Jun  7 2023, 05:45:37) [MSC v.1934 64 bit (AMD64)]


## 2. Dataset Structure Check

In [2]:
print("Directory structure:\n")
for split_name, split_dir in SPLITS.items():
    if not split_dir.exists():
        print(f"  ⚠ MISSING: {split_dir}")
        continue
    for cls in CLASSES:
        cdir = split_dir / cls
        if cdir.exists():
            n = sum(1 for f in cdir.iterdir() if f.suffix.lower() in IMG_EXT)
            print(f"  {split_name}/{cls}: {n} images")
        else:
            print(f"  ⚠ MISSING: {split_name}/{cls}")


Directory structure:

  train/NORMAL: 1341 images
  train/PNEUMONIA: 3875 images
  val/NORMAL: 8 images
  val/PNEUMONIA: 8 images
  test/NORMAL: 234 images
  test/PNEUMONIA: 390 images


## 3. Full Dataset Scan

In [3]:
import time
t0 = time.time()

records = []
for split_name, split_dir in SPLITS.items():
    for cls in CLASSES:
        cdir = split_dir / cls
        if not cdir.exists():
            continue
        for f in sorted(cdir.iterdir()):
            if f.suffix.lower() not in IMG_EXT:
                continue
            rec = {
                "split": split_name, "class": cls, "filename": f.name,
                "path": str(f), "file_size_kb": f.stat().st_size / 1024.0,
                "width": 0, "height": 0, "aspect_ratio": 0.0, "mode": "",
                "brightness": 0.0, "contrast": 0.0, "sharpness": 0.0,
                "corrupted": False, "md5": "",
            }
            try:
                img = Image.open(f)
                img.load()
                rec["width"], rec["height"] = img.size
                rec["aspect_ratio"] = round(rec["width"] / max(rec["height"], 1), 4)
                rec["mode"] = img.mode

                gray = img.convert("L")
                stat = ImageStat.Stat(gray)
                rec["brightness"] = round(stat.mean[0], 2)
                rec["contrast"] = round(stat.stddev[0], 2)

                edges = gray.filter(ImageFilter.Kernel(
                    size=(3, 3), kernel=[0,1,0,1,-4,1,0,1,0], scale=1, offset=128
                ))
                rec["sharpness"] = round(float(ImageStat.Stat(edges).stddev[0]), 2)

                img.close()
            except Exception:
                rec["corrupted"] = True

            # MD5 hash
            try:
                h = hashlib.md5()
                with open(f, "rb") as fh:
                    for chunk in iter(lambda: fh.read(65536), b""):
                        h.update(chunk)
                rec["md5"] = h.hexdigest()
            except Exception:
                pass

            records.append(rec)

df = pd.DataFrame(records)
elapsed = time.time() - t0
print(f"Scanned {len(df):,} images in {elapsed:.1f}s")
print(f"Corrupted: {df['corrupted'].sum()}")
df.head()


Scanned 5,856 images in 64.2s
Corrupted: 0


,split,class,filename,path,file_size_kb,width,height,aspect_ratio,mode,brightness,contrast,sharpness,corrupted,md5
0,train,NORMAL,IM-0115-0001.jpeg,C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-p...,850.375000,2090,1858,1.1249,L,128.91,62.30,8.52,False,20e1b8286b2d4e7a9869d42abd1cbf91
1,train,NORMAL,IM-0117-0001.jpeg,C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-p...,396.782227,1422,1152,1.2344,L,100.65,59.81,10.40,False,9299d3c610ad02c31c30469c9a4c2431
2,train,NORMAL,IM-0119-0001.jpeg,C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-p...,568.983398,1810,1434,1.2622,L,121.97,68.86,9.57,False,4b873c9f031149dccd93d51e092215ca
3,train,NORMAL,IM-0122-0001.jpeg,C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-p...,460.503906,1618,1279,1.2651,L,132.99,64.97,9.38,False,f510c0d0ff70e725449ecbdf93553a70
4,train,NORMAL,IM-0125-0001.jpeg,C:\Users\MOHAMEDKHEIR\Desktop\HMS-Graduation-p...,440.714844,1600,1125,1.4222,L,106.22,65.09,11.00,False,3c0c7a39407e411aa83b854f8f8fa8f0


## 4. Dataset Inventory & Class Distribution

In [4]:
inventory = df.groupby(["split", "class"]).size().unstack(fill_value=0)
inventory["Total"] = inventory.sum(axis=1)
inventory.loc["TOTAL"] = inventory.sum()
print(inventory.to_string())

# Save
inventory.to_csv(OUTPUT_DIR / "dataset_inventory.csv")
print(f"\nSaved: dataset_inventory.csv")


class  NORMAL  PNEUMONIA  Total
split                          
test      234        390    624
train    1341       3875   5216
val         8          8     16
TOTAL    1583       4273   5856

Saved: dataset_inventory.csv


In [5]:
# Class distribution plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Class Distribution by Split", fontsize=14)

for i, split_name in enumerate(SPLITS.keys()):
    sub = df[df["split"] == split_name]
    counts = sub["class"].value_counts()
    colors = ["#4CAF50" if c == "NORMAL" else "#F44336" for c in counts.index]
    bars = axes[i].bar(counts.index, counts.values, color=colors)
    axes[i].set_title(f"{split_name} (n={len(sub)})")
    axes[i].set_ylabel("Count")
    for bar in bars:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                     str(int(bar.get_height())), ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_distribution.png", dpi=150)
plt.show()
print("Saved: class_distribution.png")


Saved: class_distribution.png


## 5. Class Imbalance Analysis

In [6]:
for split_name in SPLITS:
    sub = df[df["split"] == split_name]
    n_normal = (sub["class"] == "NORMAL").sum()
    n_pneum = (sub["class"] == "PNEUMONIA").sum()
    total = len(sub)
    ratio = n_pneum / max(n_normal, 1)
    print(f"{split_name:6s}  NORMAL={n_normal:>5}  PNEUMONIA={n_pneum:>5}  "
          f"Ratio={ratio:.2f}:1  Pneumonia%={n_pneum/max(total,1)*100:.1f}%")

print("\n⚠ Training set imbalance is severe (2.89:1).")
print("  Recommend weighted CrossEntropyLoss or oversampling.")


train   NORMAL= 1341  PNEUMONIA= 3875  Ratio=2.89:1  Pneumonia%=74.3%
val     NORMAL=    8  PNEUMONIA=    8  Ratio=1.00:1  Pneumonia%=50.0%
test    NORMAL=  234  PNEUMONIA=  390  Ratio=1.67:1  Pneumonia%=62.5%

⚠ Training set imbalance is severe (2.89:1).
  Recommend weighted CrossEntropyLoss or oversampling.


## 6. Corrupted Image Detection

In [7]:
bad = df[df["corrupted"] == True]
if len(bad) == 0:
    print("✓ No corrupted images found.")
else:
    print(f"⚠ {len(bad)} corrupted image(s):")
    for _, r in bad.iterrows():
        print(f"  [{r['split']}/{r['class']}] {r['filename']}")


✓ No corrupted images found.


## 7. Image Dimensions Analysis

In [8]:
ok = df[~df["corrupted"]]

print("Dimension statistics:\n")
for split_name in SPLITS:
    sub = ok[ok["split"] == split_name]
    if sub.empty: continue
    print(f"  {split_name:6s}  W: {sub['width'].min():>5}–{sub['width'].max():<5} "
          f"avg={sub['width'].mean():.0f}  "
          f"H: {sub['height'].min():>5}–{sub['height'].max():<5} "
          f"avg={sub['height'].mean():.0f}")

print(f"\nUnique widths:  {ok['width'].nunique()}")
print(f"Unique heights: {ok['height'].nunique()}")
print(f"Unique (W×H):   {len(ok.groupby(['width','height']))}")


Dimension statistics:

  train   W:   384–2916  avg=1321  H:   127–2663  avg=968
  val     W:   968–1776  avg=1348  H:   592–1416  avg=1003
  test    W:   728–2752  avg=1388  H:   344–2713  avg=992

Unique widths:  867
Unique heights: 1089
Unique (W×H):   4803


In [9]:
fig, ax = plt.subplots(figsize=(10, 7))
for cls, color in [("NORMAL", "#4CAF50"), ("PNEUMONIA", "#F44336")]:
    sub = ok[ok["class"] == cls]
    ax.scatter(sub["width"], sub["height"], alpha=0.25, s=8, label=cls, color=color)
ax.set_xlabel("Width (px)")
ax.set_ylabel("Height (px)")
ax.set_title("Image Dimensions by Class")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dimension_scatter.png", dpi=150)
plt.show()
print("Saved: dimension_scatter.png")


Saved: dimension_scatter.png


## 8. Aspect Ratio Analysis

In [10]:
ars = ok["aspect_ratio"]
sq = ((ars >= 0.95) & (ars <= 1.05)).sum()
ls = (ars > 1.05).sum()
pt = (ars < 0.95).sum()
print(f"Range:     {ars.min():.3f} – {ars.max():.3f}")
print(f"Mean:      {ars.mean():.3f}   Median: {ars.median():.3f}")
print(f"Square:    {sq:>5}  ({sq/len(ars)*100:.1f}%)")
print(f"Landscape: {ls:>5}  ({ls/len(ars)*100:.1f}%)")
print(f"Portrait:  {pt:>5}  ({pt/len(ars)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 5))
for cls, c in [("NORMAL", "#4CAF50"), ("PNEUMONIA", "#F44336")]:
    ax.hist(ok[ok["class"]==cls]["aspect_ratio"], bins=40, alpha=0.55, label=cls, color=c, edgecolor="white")
ax.set_xlabel("Aspect Ratio (W/H)")
ax.set_ylabel("Count")
ax.set_title("Aspect Ratio Distribution")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "aspect_ratio_distribution.png", dpi=150)
plt.show()
print("Saved: aspect_ratio_distribution.png")


Range:     0.835 – 3.379
Mean:      1.443   Median: 1.416
Square:      123  (2.1%)
Landscape:  5709  (97.5%)
Portrait:     24  (0.4%)


Saved: aspect_ratio_distribution.png


## 9. File Size Analysis

In [11]:
for cls in CLASSES:
    sub = ok[ok["class"] == cls]
    s = sub["file_size_kb"]
    print(f"{cls:12s}  Avg={s.mean():>7.1f} KB  Median={s.median():>7.1f} KB  "
          f"Min={s.min():>5.1f}  Max={s.max():>8.1f} KB")

ratio = ok[ok["class"]=="NORMAL"]["file_size_kb"].mean() / max(ok[ok["class"]=="PNEUMONIA"]["file_size_kb"].mean(), 1)
print(f"\n⚠ NORMAL images are {ratio:.1f}× larger than PNEUMONIA on average.")
print("  This may indicate different scanners or export settings per class.")

fig, ax = plt.subplots(figsize=(10, 5))
for cls, c in [("NORMAL", "#4CAF50"), ("PNEUMONIA", "#F44336")]:
    ax.hist(ok[ok["class"]==cls]["file_size_kb"], bins=50, alpha=0.55, label=cls, color=c, edgecolor="white")
ax.set_xlabel("File Size (KB)")
ax.set_ylabel("Count")
ax.set_title("File Size Distribution by Class")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "file_size_distribution.png", dpi=150)
plt.show()
print("Saved: file_size_distribution.png")


NORMAL        Avg=  538.2 KB  Median=  506.0 KB  Min= 45.1  Max=  2357.8 KB
PNEUMONIA     Avg=   83.2 KB  Median=   69.7 KB  Min=  5.3  Max=   583.0 KB

⚠ NORMAL images are 6.5× larger than PNEUMONIA on average.
  This may indicate different scanners or export settings per class.


Saved: file_size_distribution.png


## 10. Brightness & Contrast Analysis

In [12]:
for cls in CLASSES:
    sub = ok[ok["class"] == cls]
    print(f"{cls:12s}  Brightness: mean={sub['brightness'].mean():>6.1f}  std={sub['brightness'].std():>5.1f}  "
          f"Contrast: mean={sub['contrast'].mean():>6.1f}  std={sub['contrast'].std():>5.1f}")

dark = ok[ok["brightness"] < 30]
bright = ok[ok["brightness"] > 220]
print(f"\nVery dark (<30):   {len(dark)}")
print(f"Very bright (>220): {len(bright)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, cls in enumerate(CLASSES):
    sub = ok[ok["class"] == cls]
    c = "#4CAF50" if cls == "NORMAL" else "#F44336"
    axes[i].hist(sub["brightness"], bins=40, color=c, alpha=0.85, edgecolor="white")
    axes[i].axvline(sub["brightness"].mean(), color="black", ls="--", label=f"Mean={sub['brightness'].mean():.0f}")
    axes[i].set_title(f"{cls} — Brightness")
    axes[i].set_xlabel("Mean Pixel Intensity")
    axes[i].set_ylabel("Count")
    axes[i].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "brightness_histograms.png", dpi=150)
plt.show()
print("Saved: brightness_histograms.png")


NORMAL        Brightness: mean= 122.6  std= 13.5  Contrast: mean=  61.3  std=  5.8
PNEUMONIA     Brightness: mean= 122.8  std= 19.9  Contrast: mean=  55.4  std= 10.0

Very dark (<30):   0
Very bright (>220): 1


Saved: brightness_histograms.png


In [13]:
fig, ax = plt.subplots(figsize=(8, 5))
data = [ok[ok["class"]==cls]["contrast"].values for cls in CLASSES]
bp = ax.boxplot(data, labels=CLASSES, patch_artist=True)
for patch, c in zip(bp["boxes"], ["#4CAF50", "#F44336"]):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)
ax.set_ylabel("Contrast (Pixel Std Dev)")
ax.set_title("Contrast: NORMAL vs PNEUMONIA")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "contrast_boxplot.png", dpi=150)
plt.show()
print("Saved: contrast_boxplot.png")


Saved: contrast_boxplot.png


## 11. Pixel Intensity Histograms

In [14]:
rng = np.random.RandomState(42)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, cls in enumerate(CLASSES):
    sub = ok[(ok["class"] == cls) & (ok["split"] == "train")]
    sample = sub.sample(n=min(150, len(sub)), random_state=42)
    all_px = []
    for _, row in sample.iterrows():
        try:
            img = Image.open(row["path"]).convert("L")
            all_px.append(np.array(img).ravel())
            img.close()
        except: pass
    if all_px:
        merged = np.concatenate(all_px)
        c = "#4CAF50" if cls == "NORMAL" else "#F44336"
        axes[i].hist(merged, bins=64, color=c, alpha=0.85, edgecolor="white", density=True)
        axes[i].set_title(f"{cls} — Pixel Intensity (train sample)")
        axes[i].set_xlabel("Pixel Value (0–255)")
        axes[i].set_ylabel("Density")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pixel_intensity_histograms.png", dpi=150)
plt.show()
print("Saved: pixel_intensity_histograms.png")


Saved: pixel_intensity_histograms.png


## 12. Sharpness / Blur Estimation

In [15]:
for cls in CLASSES:
    sub = ok[ok["class"] == cls]
    s = sub["sharpness"]
    print(f"{cls:12s}  mean={s.mean():>6.1f}  median={s.median():>6.1f}  min={s.min():>5.1f}  max={s.max():>6.1f}")

p5 = ok["sharpness"].quantile(0.05)
blurry = ok[ok["sharpness"] < p5]
print(f"\n5th percentile: {p5:.1f}")
print(f"Potentially blurry: {len(blurry)}")

fig, ax = plt.subplots(figsize=(10, 5))
for cls, c in [("NORMAL", "#4CAF50"), ("PNEUMONIA", "#F44336")]:
    vals = ok[ok["class"]==cls]["sharpness"]
    p99 = vals.quantile(0.99)
    ax.hist(vals.clip(upper=p99), bins=40, alpha=0.55, label=cls, color=c, edgecolor="white")
ax.set_xlabel("Sharpness")
ax.set_ylabel("Count")
ax.set_title("Sharpness Distribution by Class")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sharpness_distribution.png", dpi=150)
plt.show()
print("Saved: sharpness_distribution.png")


NORMAL        mean=  11.0  median=  10.7  min=  5.3  max=  34.8
PNEUMONIA     mean=   9.4  median=   8.9  min=  4.0  max=  49.3

5th percentile: 7.0
Potentially blurry: 290
Saved: sharpness_distribution.png


## 13. Duplicate Filename Detection

In [16]:
dup_names = df.groupby("filename").filter(lambda x: len(x) > 1)
if dup_names.empty:
    print("✓ No duplicate filenames across the dataset.")
else:
    print(f"⚠ {dup_names['filename'].nunique()} filename(s) in multiple locations:")
    for name, grp in dup_names.groupby("filename"):
        locs = ", ".join(f"{r['split']}/{r['class']}" for _, r in grp.iterrows())
        print(f"  {name} → {locs}")


✓ No duplicate filenames across the dataset.


## 14. Exact Duplicate Image Detection (MD5)

In [17]:
hashed = df[(df["md5"] != "") & (~df["corrupted"])]
dup_hashes = hashed.groupby("md5").filter(lambda x: len(x) > 1)
n_groups = dup_hashes["md5"].nunique()
n_redundant = len(dup_hashes) - n_groups

if n_groups == 0:
    print("✓ No exact duplicate images.")
else:
    print(f"⚠ {n_groups} duplicate group(s), {n_redundant} redundant image(s):\n")
    for md5, grp in dup_hashes.groupby("md5"):
        paths = [f"{r['split']}/{r['class']}/{r['filename']}" for _, r in grp.iterrows()]
        print(f"  [{len(grp)}×] {paths[0]}")
        for p in paths[1:]:
            print(f"       = {p}")


⚠ 30 duplicate group(s), 32 redundant image(s):

  [2×] test/NORMAL/NORMAL2-IM-0246-0001-0001.jpeg
       = test/NORMAL/NORMAL2-IM-0246-0001-0002.jpeg
  [2×] train/PNEUMONIA/person1619_bacteria_4267.jpeg
       = train/PNEUMONIA/person1619_bacteria_4268.jpeg
  [2×] train/PNEUMONIA/person1349_bacteria_3436.jpeg
       = train/PNEUMONIA/person1349_bacteria_3437.jpeg
  [2×] train/PNEUMONIA/person1481_bacteria_3866.jpeg
       = train/PNEUMONIA/person1481_bacteria_3867.jpeg
  [2×] train/PNEUMONIA/person357_virus_734.jpeg
       = train/PNEUMONIA/person357_virus_735.jpeg
  [2×] train/PNEUMONIA/person1261_virus_2147.jpeg
       = train/PNEUMONIA/person1261_virus_2148.jpeg
  [2×] train/PNEUMONIA/person1430_bacteria_3695.jpeg
       = train/PNEUMONIA/person1430_bacteria_3696.jpeg
  [2×] train/PNEUMONIA/person162_virus_321.jpeg
       = train/PNEUMONIA/person162_virus_322.jpeg
  [2×] test/PNEUMONIA/person128_bacteria_606.jpeg
       = test/PNEUMONIA/person128_bacteria_607.jpeg
  [2×] train/PNEU

## 15. Leakage Detection (Train ↔ Val ↔ Test)

In [18]:
split_hashes = {}
for split_name in SPLITS:
    sub = hashed[hashed["split"] == split_name]
    split_hashes[split_name] = set(sub["md5"])

pairs = [("train", "val"), ("train", "test"), ("val", "test")]
leaked = False
for s1, s2 in pairs:
    overlap = split_hashes.get(s1, set()) & split_hashes.get(s2, set())
    if overlap:
        leaked = True
        print(f"⚠ LEAKAGE: {len(overlap)} identical image(s) in {s1} AND {s2}")
    else:
        print(f"✓ {s1} ↔ {s2}: clean")

if not leaked:
    print("\n✓ No cross-split data leakage detected.")


✓ train ↔ val: clean
✓ train ↔ test: clean
✓ val ↔ test: clean

✓ No cross-split data leakage detected.


## 16. NORMAL vs PNEUMONIA — Statistical Comparison

In [19]:
comparison = ok.groupby("class").agg({
    "brightness": ["mean", "median", "std"],
    "contrast": ["mean", "median", "std"],
    "sharpness": ["mean", "median"],
    "file_size_kb": ["mean", "median"],
    "width": ["mean", "median"],
    "height": ["mean", "median"],
}).round(1)
print(comparison.to_string())


          brightness              contrast              sharpness        file_size_kb          width          height        
                mean median   std     mean median   std      mean median         mean median    mean  median    mean  median
class                                                                                                                       
NORMAL         122.6  122.4  13.5     61.3   61.5   5.8      11.0   10.7        538.2  506.0  1686.4  1654.0  1378.6  1323.0
PNEUMONIA      122.8  122.9  19.9     55.4   54.8  10.0       9.4    8.9         83.2   69.7  1195.1  1160.0   819.6   776.0


In [20]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Brightness
for cls, c in [("NORMAL", "#4CAF50"), ("PNEUMONIA", "#F44336")]:
    axes[0,0].hist(ok[ok["class"]==cls]["brightness"], bins=30, alpha=0.55, label=cls, color=c)
axes[0,0].set_title("Brightness")
axes[0,0].legend()

# Contrast
for cls, c in [("NORMAL", "#4CAF50"), ("PNEUMONIA", "#F44336")]:
    axes[0,1].hist(ok[ok["class"]==cls]["contrast"], bins=30, alpha=0.55, label=cls, color=c)
axes[0,1].set_title("Contrast")
axes[0,1].legend()

# Sharpness
for cls, c in [("NORMAL", "#4CAF50"), ("PNEUMONIA", "#F44336")]:
    vals = ok[ok["class"]==cls]["sharpness"]
    axes[1,0].hist(vals.clip(upper=vals.quantile(0.99)), bins=30, alpha=0.55, label=cls, color=c)
axes[1,0].set_title("Sharpness")
axes[1,0].legend()

# File size
for cls, c in [("NORMAL", "#4CAF50"), ("PNEUMONIA", "#F44336")]:
    axes[1,1].hist(ok[ok["class"]==cls]["file_size_kb"], bins=30, alpha=0.55, label=cls, color=c)
axes[1,1].set_title("File Size (KB)")
axes[1,1].legend()

plt.suptitle("NORMAL vs PNEUMONIA — Feature Comparison", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_comparison.png", dpi=150)
plt.show()
print("Saved: class_comparison.png")


Saved: class_comparison.png


## 17. Sample Image Grids

In [21]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle("Sample Chest X-Ray Images (Train Set)", fontsize=14)

for row, cls in enumerate(CLASSES):
    sub = ok[(ok["class"] == cls) & (ok["split"] == "train")].head(5)
    for col, (_, rec) in enumerate(sub.iterrows()):
        try:
            img = Image.open(rec["path"]).convert("L")
            axes[row, col].imshow(np.array(img), cmap="gray")
            axes[row, col].set_title(f"{cls}\n{rec['width']}×{rec['height']}", fontsize=8)
            img.close()
        except: pass
        axes[row, col].axis("off")
    for col in range(len(sub), 5):
        axes[row, col].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_grid.png", dpi=150)
plt.show()
print("Saved: sample_grid.png")


Saved: sample_grid.png


## 18. Export Metadata CSV

In [22]:
csv_cols = ["split", "class", "filename", "width", "height", "aspect_ratio",
            "file_size_kb", "mode", "brightness", "contrast", "sharpness", "corrupted"]
df[csv_cols].to_csv(OUTPUT_DIR / "image_metadata.csv", index=False)
print(f"Saved: image_metadata.csv ({len(df)} rows)")


Saved: image_metadata.csv (5856 rows)


## 19. Final EDA Summary & Recommendations

In [23]:
ok = df[~df["corrupted"]]
n_total = len(df)
n_corrupt = df["corrupted"].sum()
n_dups = n_redundant if 'n_redundant' in dir() else 0
n_unique_w = ok["width"].nunique()
ratio_train = (ok[ok["split"]=="train"]["class"]=="PNEUMONIA").sum() / max((ok[ok["split"]=="train"]["class"]=="NORMAL").sum(), 1)
val_n = (ok["split"]=="val").sum()
fs_normal = ok[ok["class"]=="NORMAL"]["file_size_kb"].mean()
fs_pneum = ok[ok["class"]=="PNEUMONIA"]["file_size_kb"].mean()
fs_ratio = fs_normal / max(fs_pneum, 1)
n_rgb = (ok["mode"] == "RGB").sum()
n_bright = (ok["brightness"] > 220).sum()

summary = f"""# Chest X-Ray EDA Summary

## Dataset
- Total images: {n_total:,}
- Corrupted: {n_corrupt}
- Train: {(df['split']=='train').sum():,} | Val: {(df['split']=='val').sum()} | Test: {(df['split']=='test').sum()}

## Key Findings
- Class imbalance: {ratio_train:.2f}:1 (PNEUMONIA:NORMAL in train)
- Validation set: {val_n} images (critically small)
- Exact duplicates: {n_dups} redundant images
- Cross-split leakage: None detected
- Unique dimensions: {n_unique_w} widths
- RGB images: {n_rgb} (mixed with grayscale)
- File size gap: NORMAL avg {fs_normal:.0f} KB vs PNEUMONIA {fs_pneum:.0f} KB ({fs_ratio:.1f}x)
- Overexposed (>220): {n_bright}

## Risks
1. Severe class imbalance (2.89:1) — use weighted loss
2. Validation set too small (16 images) — carve 10% from train
3. File size disparity may cause spurious correlation
4. Mixed color modes need standardization
5. Variable dimensions require mandatory resize

## Recommendations
1. Remove duplicate images
2. Expand validation split (stratified 10% from train)
3. Use weighted CrossEntropyLoss
4. Resize(256) + CenterCrop(224)
5. Convert all to RGB
6. Use F1/AUC-ROC, not accuracy
7. Apply moderate augmentation (flip, rotation, brightness jitter)

## ML Readiness Score: 6.5 / 10
Suitable for research/prototyping after preprocessing.
"""

with open(OUTPUT_DIR / "eda_summary.md", "w", encoding="utf-8") as f:
    f.write(summary)

print(summary)
print(f"\nSaved: eda_summary.md")


# Chest X-Ray EDA Summary

## Dataset
- Total images: 5,856
- Corrupted: 0
- Train: 5,216 | Val: 16 | Test: 624

## Key Findings
- Class imbalance: 2.89:1 (PNEUMONIA:NORMAL in train)
- Validation set: 16 images (critically small)
- Exact duplicates: 32 redundant images
- Cross-split leakage: None detected
- Unique dimensions: 867 widths
- RGB images: 283 (mixed with grayscale)
- File size gap: NORMAL avg 538 KB vs PNEUMONIA 83 KB (6.5x)
- Overexposed (>220): 1

## Risks
1. Severe class imbalance (2.89:1) — use weighted loss
2. Validation set too small (16 images) — carve 10% from train
3. File size disparity may cause spurious correlation
4. Mixed color modes need standardization
5. Variable dimensions require mandatory resize

## Recommendations
1. Remove duplicate images
2. Expand validation split (stratified 10% from train)
3. Use weighted CrossEntropyLoss
4. Resize(256) + CenterCrop(224)
5. Convert all to RGB
6. Use F1/AUC-ROC, not accuracy
7. Apply moderate augmentation (flip, rot

---
## Output Files

All saved to `ml/eda/outputs/`:
- `dataset_inventory.csv`
- `image_metadata.csv`
- `eda_summary.md`
- `class_distribution.png`
- `dimension_scatter.png`
- `brightness_histograms.png`
- `contrast_boxplot.png`
- `pixel_intensity_histograms.png`
- `sharpness_distribution.png`
- `aspect_ratio_distribution.png`
- `file_size_distribution.png`
- `class_comparison.png`
- `sample_grid.png`
